# Bipartite Graph Creation

#### Run Notebooks

In [1]:
%%capture
# Run data wrangling nb
%run data_grace.ipynb

#### Imports

In [2]:

# Specific to network purposes
import networkx as nx
from networkx.algorithms import bipartite
import torch

#### Data Arrangement

In [3]:
# Create df from rsvp data but drop group_id column
rsvps_df=rsvps_df.copy()
member_event=rsvps_df.drop(columns=['group_id'])


In [4]:
# Find all the unique event ids in the events df
event_nodes = events_df.event_id.unique()

# Create a dictionary to map event ids to integers
event_nodes_dict = {v: i for i, v in enumerate(event_nodes)}

# Apply the map to the member_event df and drop the original event id column
member_event['event_id_int'] = member_event['event_id'].map(event_nodes_dict)
member_event=member_event.drop(columns=['event_id'])

# Apply same map to the events df and set the new event id column as the index
events_df['event_id_int'] = events_df['event_id'].map(event_nodes_dict)
events_df.set_index('event_id_int', inplace=True,drop=True)

In [5]:
# Create a dictionary of dictionaries in order to eventually assign as node attributes for members
    # Pull from one-hot encoded members df
mem_dict = mem_df_new.to_dict(orient='index')

# Create a dictionary of dictionaries in order to eventually assign as node attributes for events
event_dict= events_df.to_dict(orient='index')

#### Bipartite Graph Creation

In [6]:
# Define empty graph
G_bipartite = nx.Graph()

# Create arrays to hold nodes
member_nodes =[]
event_nodes = []

# Add member nodes
for member_id, attrs in mem_dict.items():
    G_bipartite.add_node(member_id, type='member', **attrs)
    member_nodes.append(member_id)

# Add event nodes 
for event_id, attrs in event_dict.items():
    G_bipartite.add_node(event_id, type='event', **attrs)
    event_nodes.append(event_id)

# Add edges
for _,row in member_event.iterrows():
    G_bipartite.add_edge(row['member_id'], row['event_id_int'])

In [7]:
print(G_bipartite)


Graph with 44280 nodes and 126813 edges


In [8]:
A_bipartite = bipartite.biadjacency_matrix(G_bipartite, row_order=member_nodes, column_order=event_nodes)
A_bipartite_tensor = torch.tensor(A_bipartite.toarray(), dtype=torch.float)

In [9]:
A_bipartite_tensor

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [20]:
import os
import pandas as pd

# 1. Cast node attributes
for node, data in G_bipartite.nodes(data=True):
    for key, value in data.items():
        if isinstance(value, pd.Timestamp):
            data[key] = value.isoformat()

# 2. Cast edge attributes
for u, v, data in G_bipartite.edges(data=True):
    for key, value in data.items():
        if isinstance(value, pd.Timestamp):
            data[key] = value.isoformat()

# 3. Cast graph-level attributes
for key, value in G_bipartite.graph.items():
    if isinstance(value, pd.Timestamp):
        G_bipartite.graph[key] = value.isoformat()
nx.write_graphml(G_bipartite, os.path.join(os.getcwd(),"notebooks","data","G_bipartite.graphml"))